<a href="https://colab.research.google.com/github/MohammedShahad7/Data-Science-/blob/main/assignment%20Detecting%20Outliers%20in%20Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Retail Transactions
retail_amt = np.random.normal(loc=1750, scale=500, size=100)
retail_amt = np.clip(retail_amt, 500, 3000)

# Wholesale Transactions
wholesale_amt = np.random.uniform(40000, 80000, size=8)

# Extreme Outliers (Errors)
error_amt = [350000, 420000]

all_amounts = np.concatenate([retail_amt, wholesale_amt, error_amt])
np.random.shuffle(all_amounts)

df = pd.DataFrame({
    'Transaction_ID': [f'TXN_{i+1000}' for i in range(len(all_amounts))],
    'Customer_ID': [f'CUST_{np.random.randint(1, 50)}' for _ in range(len(all_amounts))],
    'Transaction_Amount': all_amounts.round(2),
    'Customer_Type': ['Retail' if x < 10000 else 'Wholesale' for x in all_amounts],
    'Payment_Method': np.random.choice(['Credit Card', 'UPI', 'Net Banking'], size=len(all_amounts)),
    'Order_Date': pd.date_range(start='2023-10-01', periods=len(all_amounts), freq='D')
})

df.head()

,Transaction_ID,Customer_ID,Transaction_Amount,Customer_Type,Payment_Method,Order_Date
0,TXN_1000,CUST_21,1221.14,Retail,Net Banking,2023-10-01
1,TXN_1001,CUST_48,2021.28,Retail,UPI,2023-10-02
2,TXN_1002,CUST_20,1010.74,Retail,Net Banking,2023-10-03
3,TXN_1003,CUST_8,1870.98,Retail,Credit Card,2023-10-04
4,TXN_1004,CUST_7,2482.82,Retail,Credit Card,2023-10-05


In [44]:
from scipy.stats import zscore


# =========================
# TASK 1A - Z SCORE
# =========================

df['z_score'] = zscore(df['Transaction_Amount'])

z_outliers = df[df['z_score'].abs() > 3]

z_count = len(z_outliers)
z_max = z_outliers['Transaction_Amount'].max()

z_wholesale_flagged = (
    ((z_outliers['Transaction_Amount'] >= 40000) &
     (z_outliers['Transaction_Amount'] <= 80000))
    .any()
)

print("Z-Score Results")
print("Number of outliers detected:", z_count)
print("Maximum transaction amount detected:", z_max)
print("Whether Wholesale transactions were flagged:", z_wholesale_flagged)


# =========================
# TASK 1B - IQR
# =========================

Q1 = df['Transaction_Amount'].quantile(0.25)
Q3 = df['Transaction_Amount'].quantile(0.75)

IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

iqr_outliers = df[
    (df['Transaction_Amount'] < lower_fence) |
    (df['Transaction_Amount'] > upper_fence)
]

iqr_count = len(iqr_outliers)
iqr_max = iqr_outliers['Transaction_Amount'].max()

iqr_wholesale_flagged = (
    ((iqr_outliers['Transaction_Amount'] >= 40000) &
     (iqr_outliers['Transaction_Amount'] <= 80000))
    .any()
)

print("\nIQR Results")
print("Number of outliers detected:", iqr_count)
print("Maximum transaction amount detected:", iqr_max)
print("Whether Wholesale transactions were flagged:", iqr_wholesale_flagged)


# =========================
# TASK 2 - COMPARISON TABLE
# =========================

comparison_table = pd.DataFrame({
    "Method": ["Z-Score", "IQR"],
    "No_of_Outliers": [z_count, iqr_count],
    "Max_Outlier_Amount": [z_max, iqr_max],
    "Wholesale_Flagged_(True/False)": [
        z_wholesale_flagged,
        iqr_wholesale_flagged
    ]
})

print("\nComparison Table")
print(comparison_table)


# =========================
# TASK 3 - SANITIZATION POLICY
# =========================

df_cleaned = df.copy()


# Remove transactions above ₹2,50,000
df_cleaned = df_cleaned[
    df_cleaned['Transaction_Amount'] <= 250000
].copy()



# Cap wholesale transactions at ₹80,000
mask = (
    (df_cleaned['Customer_Type'] == 'Wholesale') &
    (df_cleaned['Transaction_Amount'] > 80000)
)



# =========================
# BEFORE VS AFTER TABLE
# =========================

summary_table = pd.DataFrame({
    "Metric": ["Mean", "Max"],
    "Before_Cleaning": [
        df['Transaction_Amount'].mean(),
        df['Transaction_Amount'].max()
    ],
    "After_Cleaning": [
        df_cleaned['Transaction_Amount'].mean(),
        df_cleaned['Transaction_Amount'].max()
    ]
})

print("\nSummary Table")
print(summary_table)

Z-Score Results
Number of outliers detected: 2
Maximum transaction amount detected: 420000.0
Whether Wholesale transactions were flagged: False

IQR Results
Number of outliers detected: 10
Maximum transaction amount detected: 420000.0
Whether Wholesale transactions were flagged: True

Comparison Table
    Method  No_of_Outliers  Max_Outlier_Amount  Wholesale_Flagged_(True/False)
0  Z-Score               2            420000.0                           False
1      IQR              10            420000.0                            True

Summary Table
  Metric  Before_Cleaning  After_Cleaning
0   Mean     12756.948909     5863.559074
1    Max    420000.000000    77716.390000
